# Building a RAG system for web data using ollama service with llama3.2:1b

## What is Large Language Models (LLM)?
LLM are a category of foundation models trained on immense amounts of data making them capable of understanding and generating natural language and other types of content to perform a wide range of tasks.

## What is LangChain?
LangChain is an open source orchestration framework for the development of applications using LLMs. Available in both Python- and JavaScript-based libraries, LangChain’s tools and APIs simplify the process of building LLM-driven applications like chatbots and virtual agents. 

## What is Retrieval Augmented Generation (RAG)?
RAG is a technique in natural language processing (NLP) that combines information retrieval and generative models to produce more accurate, relevant, and contextually aware responses.

## Setup

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 7.06 ms, sys: 7.83 ms, total: 14.9 ms
Wall time: 1.21 s


### Lib install

In [2]:
%pip install urllib3 beautifulsoup4 sentence_transformers langchain langchain_chroma langchain-community langchain-huggingface unstructured langchain-ollama ollama  ipython-autotime

Note: you may need to restart the kernel to use updated packages.


## Index the URLs to create the knowledge base

In [3]:
URLS_DICTIONARY = {
    "aws_rag_page": "https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/",
    "azure_rag_page": "https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652",
}
COLLECTION_NAME = "my_collection"
documents = []

In [4]:
class Document:
    def __init__(self, metadata, page_content):
        self.metadata = metadata
        self.page_content = page_content

In [5]:
import requests
import urllib3
from urllib3.exceptions import InsecureRequestWarning
from bs4 import BeautifulSoup
import re

# Suppress SSL warnings for development
urllib3.disable_warnings(InsecureRequestWarning)


def read_url_clean_text(url: str, verify_ssl: bool = False) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        # Try with SSL verification first
        response = requests.get(url, headers=headers, verify=verify_ssl, timeout=30)
        response.raise_for_status()
    except requests.exceptions.SSLError:
        print(f"SSL verification failed for {url}, trying without verification...")
        # Fallback without SSL verification
        response = requests.get(url, headers=headers, verify=False, timeout=30)
        response.raise_for_status()

    # Parse HTML and extract clean text
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove unwanted elements
    for element in soup(
        ["script", "style", "nav", "header", "footer", "aside", "noscript"]
    ):
        element.decompose()

    # Get text content
    text = soup.get_text()

    # Clean up whitespace and formatting
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    text = " ".join(chunk for chunk in chunks if chunk)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [6]:
for name, url in URLS_DICTIONARY.items():
    print(f"Loading from {url}")
    response = read_url_clean_text(url, verify_ssl=False)

    if len(response) > 0:
        data = {
            "metadata": {"source": url, "name": name},
            "page_content": response,
        }

        documents.append(
            Document(metadata=data["metadata"], page_content=data["page_content"])
        )
        print(f"Loaded from {url}")
    else:
        print(f"Failed to retrieve content from {url}")


print(documents[0].metadata)
print(documents[0].page_content)

Loading from https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/
Loaded from https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/
Loading from https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652
Loaded from https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652
{'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/', 'name': 'aws_rag_page'}
Optimize RAG in production environments using Amazon SageMaker JumpStart and Amazon OpenSearch Service | Artificial Intelligence Skip to Main Content Artificial Intelligence Optimize RAG in production environments using Amazon SageMaker JumpStart a

In [7]:
len(documents)

2

In [8]:
documents[0].metadata

{'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/',
 'name': 'aws_rag_page'}

In [9]:
doc_id = 0
for doc in documents:
    doc.page_content = " ".join(doc.page_content.split())  # remove white space

    doc.metadata["id"] = (
        doc_id  # make a document id and add it to the document metadata
    )

    print(doc.metadata)
    doc_id += 1

# Let's see how our sample document looks now after we cleaned it up.
display(documents[1].metadata)
display(documents[1].page_content)

{'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/', 'name': 'aws_rag_page', 'id': 0}
{'source': 'https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652', 'name': 'azure_rag_page', 'id': 1}


{'source': 'https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652',
 'name': 'azure_rag_page',
 'id': 1}

'Bonus Journey: Agentic RAG - Combining Agents with Retrieval-Augmented GenerationBlog PostAI - Azure AI services Blog 6 MIN READBonus RAG Time Journey: Agentic RAGMattGotteinerMicrosoftApr 16, 2025This is a bonus post for RAG Time, a 6-part educational series on retrieval-augmented generation (RAG). In this series, we explored topics such as indexing and retrieval techniques for RAG, data ingestion, and storage optimization. The final topic for this series covers agentic RAG, and how to use semi-autonomous agents to make a dynamic and self-refining retrieval system. What we\'ll cover: Overview and definition of agentic RAG Example of a single-shot RAG flow Two examples of agentic RAG: single-step and multi-step reflection What is agentic RAG? An agent is a component of an AI application that leverages generative models to make decisions and execute actions autonomously. Agentic RAG improves the traditional RAG flow by actively interacting with its environment using tools, memory, and 

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
docs

[Document(metadata={'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/', 'name': 'aws_rag_page', 'id': 0}, page_content='Optimize RAG in production environments using Amazon SageMaker JumpStart and Amazon OpenSearch Service | Artificial Intelligence Skip to Main Content Artificial Intelligence Optimize RAG in production environments using Amazon SageMaker JumpStart and Amazon OpenSearch Service Generative AI has revolutionized customer interactions across industries by offering personalized, intuitive experiences powered by unprecedented access to information. This transformation is further enhanced by Retrieval Augmented'),
 Document(metadata={'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/', 'name': 'aws_rag_page', 'id': 0}, page_content='Generation (RAG), a techni

### HuggingFace Embeddings

In [11]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# Initialize HuggingFace embeddings with a popular model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Vector Store
Let's load the content into a local instance of a vector database, using Chromadb.

In [12]:
from langchain.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)
vectorstore

## Query 
Let's do a quick search of our vector database to test it out!

In [13]:
test1_query = "What is Sagemaker performance?"

test1_search_result = vectorstore.similarity_search_with_score(test1_query, k=4)
test1_search_result

[(Document(metadata={'name': 'aws_rag_page', 'id': 0, 'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/'}, page_content='to a SageMaker real-time endpoint: sagemaker.jumpstart.model JumpStartModel model_id "meta-textgeneration-llama-3-8b-instruct" accept_eula model JumpStartModel(model_idmodel_id) llm_predictor modeldeploy(accept_eulaaccept_eula) model_id "huggingface-sentencesimilarity-bge-large-en-v1-5" text_embedding_model JumpStartModel(model_idmodel_id) embedding_predictor text_embedding_modeldeploy() Content handlers are crucial for formatting data for SageMaker endpoints. They transform inputs into the format expected'),
  0.9224939346313477),
 (Document(metadata={'name': 'aws_rag_page', 'id': 0, 'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/'}, page_content

In [14]:
test2_query = "Share examples of agentic RAG?"

test2_search_result = vectorstore.similarity_search_with_score(test2_query, k=4)
test2_search_result

[(Document(metadata={'source': 'https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652', 'id': 1, 'name': 'azure_rag_page'}, page_content='generated, considering that there’s missing information searches couldn’t find. The agentic RAG loop continues until the answer is of sufficient quality or too much time has passed. Single-Step Reflection We can put all the components of agentic RAG together into our first sample implementation: single-step reflection. The single-shot RAG flow is run to get a candidate answer. The answer is evaluated using relevance and groundedness evaluators. If both scores from these evaluators are at least 4, the'),
  0.8130398988723755),
 (Document(metadata={'id': 1, 'name': 'azure_rag_page', 'source': 'https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652'}, page_content="agents to make a dynamic and self-refining retrieval system. What we'll cover: Overvie

## Set up a retriever

The retrieved information from the vector store serves as additional context or knowledge that can be used by a generative model. Lets specify search kwargs like k (the number of documents to return (Default: 4)) to use when doing retrieval.

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x3451c7c10>, search_kwargs={'k': 4})

## Generate a response with a generative model

Finally, we’ll generate a response. The generative model (llama3.2:1b) uses the retrieved information to produce a more accurate and contextually relevant response to the questions.

In [16]:
from langchain_ollama.llms import OllamaLLM

my_ollama_model = "llama3.2"

llm_client = OllamaLLM(
    model=my_ollama_model,
    base_url="http://localhost:11434",
    headers={"Content-Type": "application/json"},
)

llm_client

OllamaLLM(model='llama3.2', base_url='http://localhost:11434')

In [17]:
from langchain_core.prompts import ChatPromptTemplate

# Create a ChatPromptTemplate for the RAG system
template = """Generate a summary of the context that answers the question. Explain the answer in multiple steps if possible. 
Answer style should match the context. Ideal Answer Length 2-3 sentences.\n\n{context}\nQuestion: {question}\nAnswer:
"""

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant that answers questions based on the provided context. Use only the information from the context to answer the question. If the context doesn't contain enough information to answer the question, say so.",
        ),
        ("human", "Context: {context}\n\nQuestion: {question}"),
    ]
)

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Generate a summary of the context that answers the question. Explain the answer in multiple steps if possible. \nAnswer style should match the context. Ideal Answer Length 2-3 sentences.\n\n{context}\nQuestion: {question}\nAnswer:\n'), additional_kwargs={})])

In [18]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

In [19]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create the RAG chain
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_client
    | StrOutputParser()
)

chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x3451c7c10>, search_kwargs={'k': 4})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Generate a summary of the context that answers the question. Explain the answer in multiple steps if possible. \nAnswer style should match the context. Ideal Answer Length 2-3 sentences.\n\n{context}\nQuestion: {question}\nAnswer:\n'), additional_kwargs={})])
| OllamaLLM(model='llama3.2', base_url='http://localhost:11434')
| StrOutputParser()

## TEST to ask Questions

In [20]:
print("Test question: ", test1_query)

test1_response = chain.invoke(test1_query)

print("Test1 response: ", test1_response)

Test question:  What is Sagemaker performance?
Test1 response:  SageMaker performance refers to the efficiency, scalability, and reliability of Amazon SageMaker's machine learning (ML) platform. Here's a step-by-step explanation:

1. **Efficiency**: SageMaker provides optimized computing environments for ML workflows, reducing training and deployment times.
2. **Scalability**: The platform supports large-scale deployments of models across multiple instances, making it suitable for high-traffic applications.
3. **Reliability**: SageMaker features automated deployment, rollbacks, and monitoring, ensuring consistent performance and minimizing downtime.

By leveraging these aspects, SageMaker enables businesses to build and deploy scalable, reliable, and efficient ML models, driving better performance and outcomes in their RAG applications.


In [21]:
import pprint

pprint.pprint(test1_response)

('SageMaker performance refers to the efficiency, scalability, and reliability '
 "of Amazon SageMaker's machine learning (ML) platform. Here's a step-by-step "
 'explanation:\n'
 '\n'
 '1. **Efficiency**: SageMaker provides optimized computing environments for '
 'ML workflows, reducing training and deployment times.\n'
 '2. **Scalability**: The platform supports large-scale deployments of models '
 'across multiple instances, making it suitable for high-traffic '
 'applications.\n'
 '3. **Reliability**: SageMaker features automated deployment, rollbacks, and '
 'monitoring, ensuring consistent performance and minimizing downtime.\n'
 '\n'
 'By leveraging these aspects, SageMaker enables businesses to build and '
 'deploy scalable, reliable, and efficient ML models, driving better '
 'performance and outcomes in their RAG applications.')


In [22]:
print("Test question: ", test2_query)

test2_response = chain.invoke(test2_query)

print("Test2 response: ", test2_response)

Test question:  Share examples of agentic RAG?
Test2 response:  Here is a summary of the context in 2-3 sentences:

Agentic RAG (Reactive Active Generation) is an AI application that leverages generative models to make decisions and execute actions autonomously, improving traditional retrieval systems into self-refining solutions. It combines tools, memory, and secure access to data with LLM-based evaluators to assess relevance and factual groundedness of generated answers. Two examples of agentic RAG are single-step reflection and multi-step reflection, which involve a continuous loop of evaluation and improvement.

Here's an explanation in multiple steps:

1. **Understanding Agentic RAG**: Agentic RAG is a type of AI application that uses generative models to make decisions and execute actions autonomously.
2. **Key Characteristics**: It combines tools, memory, and secure access to data with LLM-based evaluators to assess relevance and factual groundedness of generated answers.
3. **

In [23]:
pprint.pprint(test2_response)

('Here is a summary of the context in 2-3 sentences:\n'
 '\n'
 'Agentic RAG (Reactive Active Generation) is an AI application that leverages '
 'generative models to make decisions and execute actions autonomously, '
 'improving traditional retrieval systems into self-refining solutions. It '
 'combines tools, memory, and secure access to data with LLM-based evaluators '
 'to assess relevance and factual groundedness of generated answers. Two '
 'examples of agentic RAG are single-step reflection and multi-step '
 'reflection, which involve a continuous loop of evaluation and improvement.\n'
 '\n'
 "Here's an explanation in multiple steps:\n"
 '\n'
 '1. **Understanding Agentic RAG**: Agentic RAG is a type of AI application '
 'that uses generative models to make decisions and execute actions '
 'autonomously.\n'
 '2. **Key Characteristics**: It combines tools, memory, and secure access to '
 'data with LLM-based evaluators to assess relevance and factual groundedness '
 'of generated a